# Step 3 — Experiment Tracking with MLflow
In Step 2, results lived only in notebook outputs: re-run a cell and the old numbers are gone.
Here, **every training run is recorded permanently in MLflow** — what was trained, with which settings, on which data,
how well it scored, and the plots and model it produced. Runs can be compared side by side in a web UI, and the chosen model
is registered so later steps (API, Docker, CI/CD) can load it by name.

### MLflow vocabulary
| Term | Meaning | In this project |
|---|---|---|
| **Experiment** | A folder grouping related runs | `heart-disease-classification` |
| **Run** | One training attempt | e.g. "Random Forest – tuned" |
| **Parameters** | Inputs/settings of a run | `C=1`, `max_depth=8`, `drop_sparse=False` |
| **Metrics** | Numeric results | `cv_roc_auc_mean = 0.91` |
| **Artifacts** | Files saved with a run | plots, CSV tables, the model itself |
| **Tags** | Labels for filtering | `stage=tuning`, `model_type=random_forest` |
| **Nested runs** | Child runs under a parent | each grid-search combination under "tuning" |
| **Model Registry** | Versioned catalogue of approved models | `heart-disease-classifier`, alias `champion` |

### What gets logged
```
Experiment: heart-disease-classification
 ├─ feature-set test:  keep vs drop ca/thal/slope        (2 runs)
 ├─ baselines:         LR, RF with defaults               (2 runs, full logging)
 ├─ tuning – LR        parent run ─┬─ 8 child runs        (one per setting tried)
 │                                 └─ tuned LR             (full logging + test-set results)
 └─ tuning – RF        parent run ─┬─ 36 child runs
                                   └─ tuned RF             (full logging + test-set results)
                       → best model registered as heart-disease-classifier @champion
```

## 1. Setup

In [ ]:
import os
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"   # silence an unrelated MLflow startup hint

import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import seaborn as sns
from mlflow import MlflowClient
from mlflow.models import infer_signature
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, precision_score,
                             recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import (GridSearchCV, StratifiedKFold, cross_val_predict,
                                     cross_validate, train_test_split)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA_DIR = ROOT / "data"
SEED = 42

BLUE, ORANGE, GREY = "#2a78d6", "#eb6834", "#52514e"
sns.set_theme(style="whitegrid")
print("MLflow version:", mlflow.__version__)

### Point MLflow at a storage location
MLflow needs two places to store things:
- **Tracking database** `mlflow.db` (a small SQLite file) — run names, parameters, metrics, tags, registry entries.
- **Artifact folder** `mlruns/` — the files: plots, CSVs, saved models.

Both live in the **project root**, so the notebook and the MLflow UI (started from the project root) see the same data.

In [ ]:
mlflow.set_tracking_uri(f"sqlite:///{ROOT / 'mlflow.db'}")

EXPERIMENT = "heart-disease-classification"
if mlflow.get_experiment_by_name(EXPERIMENT) is None:
    mlflow.create_experiment(EXPERIMENT, artifact_location=(ROOT / "mlruns").as_uri())
mlflow.set_experiment(EXPERIMENT)

REGISTERED_MODEL = "heart-disease-classifier"
print("Tracking URI:", mlflow.get_tracking_uri())

## 2. Load data, define features, split (same rules as Steps 1–2)

In [ ]:
COLUMNS = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
           "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num"]
SOURCES = {
    "cleveland":   "processed.cleveland.data",
    "hungary":     "processed.hungarian.data",
    "switzerland": "processed.switzerland.data",
    "va":          "processed.va.data",
}

frames = []
for source, filename in SOURCES.items():
    path = DATA_DIR / filename
    assert path.exists(), f"Missing file: {path}"
    raw = pd.read_csv(path, header=None, na_values="?")
    assert raw.shape[1] == len(COLUMNS), f"{filename}: expected 14 columns, found {raw.shape[1]}"
    raw.columns = COLUMNS
    raw["source"] = source
    frames.append(raw)

df = pd.concat(frames, ignore_index=True)
df["target"] = (df["num"] > 0).astype(int)
df = df.drop(columns="num")
df[["chol", "trestbps"]] = df[["chol", "trestbps"]].replace(0, np.nan)

NUMERIC = ["age", "trestbps", "chol", "thalach", "oldpeak", "ca"]
CATEGORICAL = ["cp", "restecg", "slope", "thal"]
BINARY = ["sex", "fbs", "exang"]
FEATURES = NUMERIC + CATEGORICAL + BINARY
SPARSE = ["ca", "thal", "slope"]

X = df[FEATURES].astype(float)   # all features as floats, so missing values (NaN) are always allowed
y = df["target"]
strata = df["source"] + "_" + df["target"].astype(str)
X_train, X_test, y_train, y_test, strata_train, _ = train_test_split(
    X, y, strata, test_size=0.20, stratify=strata, random_state=SEED)

folds = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(X_train, strata_train))
print(f"Development: {len(X_train)} rows | Test: {len(X_test)} rows")

In [ ]:
def build_preprocessor(drop_sparse=False):
    num = [c for c in NUMERIC if not (drop_sparse and c in SPARSE)]
    cat = [c for c in CATEGORICAL if not (drop_sparse and c in SPARSE)]
    return ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median", add_indicator=True)),
                          ("scale", StandardScaler())]), num),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent", add_indicator=True)),
                          ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat),
        ("bin", SimpleImputer(strategy="most_frequent"), BINARY),
    ])

def make_pipeline(model, drop_sparse=False):
    return Pipeline([("prep", build_preprocessor(drop_sparse)), ("model", model)])

## 3. Logging helpers
Three small functions do all the MLflow work, so every run is logged the **same way**:

- `common_context()` — information every run should carry: data fingerprint, git commit, split settings.
- `make_plots()` — ROC curve + confusion matrix from **out-of-fold** predictions (each patient predicted by a model that never saw them during CV — no test set needed), plus feature importance.
- `log_full_run()` — runs cross-validation and logs **parameters, metrics, plots, a CV table and the model**.

*Note on `skops_trusted_types`:* recent MLflow versions save scikit-learn models in the safe **skops** format, which refuses
to store any object type it hasn't been told is safe. Two harmless internals need listing: the one-hot encoder's `numpy.dtype` and Random Forest's tree structure (`sklearn.tree._tree.Tree`).

In [ ]:
def git_commit():
    """Current git commit, so every run can be traced back to the exact code version."""
    try:
        return subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=ROOT,
                                       text=True, stderr=subprocess.DEVNULL).strip()
    except Exception:
        return "unknown"

TRAIN_DF = X_train.assign(target=y_train)
DATASET = mlflow.data.from_pandas(TRAIN_DF, targets="target", name="heart-disease-dev-set")

def common_context(drop_sparse):
    """Params and tags shared by all runs: data lineage, split settings, code version."""
    mlflow.log_params({"seed": SEED, "test_size": 0.20, "cv_folds": len(folds),
                       "drop_sparse": drop_sparse, "n_train_rows": len(X_train)})
    mlflow.set_tags({"git_commit": git_commit(), "data_digest": DATASET.digest})
    mlflow.log_input(DATASET, context="training")

def model_params(pipeline):
    """Hyperparameters of the model step only (keeps the params list readable)."""
    return {k: v for k, v in pipeline.named_steps["model"].get_params().items()
            if not isinstance(v, (dict, list)) and v is not None}

In [ ]:
SCORING = ["accuracy", "precision", "recall", "roc_auc"]
TRUSTED_TYPES = ["numpy.dtype", "sklearn.tree._tree.Tree"]   # safe internals of our encoder and trees

def make_plots(pipeline, oof_proba, title):
    """Return (roc+confusion figure, feature-importance figure) for a fitted pipeline."""
    fig1, axes = plt.subplots(1, 2, figsize=(11, 4.2))
    fpr, tpr, _ = roc_curve(y_train, oof_proba)
    axes[0].plot(fpr, tpr, color=BLUE, linewidth=2, label=f"AUC {roc_auc_score(y_train, oof_proba):.3f}")
    axes[0].plot([0, 1], [0, 1], linestyle="--", color=GREY, linewidth=1)
    axes[0].set(xlabel="False positive rate", ylabel="True positive rate (recall)", title="ROC (out-of-fold)")
    axes[0].legend(frameon=False, loc="lower right")
    ConfusionMatrixDisplay.from_predictions(y_train, (oof_proba >= 0.5).astype(int), ax=axes[1],
                                            cmap="Blues", colorbar=False,
                                            display_labels=["No disease", "Disease"])
    axes[1].set_title("Confusion matrix (out-of-fold)")
    axes[1].grid(False)
    fig1.suptitle(title, fontweight="bold")
    fig1.tight_layout()

    names = pipeline.named_steps["prep"].get_feature_names_out()
    model = pipeline.named_steps["model"]
    values = model.coef_[0] if hasattr(model, "coef_") else model.feature_importances_
    s = pd.Series(values, index=names)
    s = s.reindex(s.abs().sort_values().index)[-12:]
    fig2, ax = plt.subplots(figsize=(7, 4.5))
    ax.barh(s.index, s.values, color=[ORANGE if v > 0 else BLUE for v in s.values])
    ax.axvline(0, color=GREY, linewidth=1)
    ax.set_title(f"{title}: top features", fontweight="bold")
    ax.grid(axis="y", visible=False)
    fig2.tight_layout()
    return fig1, fig2

def log_full_run(pipeline, run_name, tags, drop_sparse, nested=False):
    """Cross-validate, then log params, metrics, plots, CV table and the fitted model. Returns run_id."""
    with mlflow.start_run(run_name=run_name, nested=nested) as run:
        common_context(drop_sparse)
        mlflow.set_tags(tags)
        mlflow.log_params(model_params(pipeline))

        # Metrics: mean and std over the 5 folds
        cv = cross_validate(pipeline, X_train, y_train, cv=folds, scoring=SCORING, n_jobs=-1)
        for m in SCORING:
            mlflow.log_metric(f"cv_{m}_mean", cv[f"test_{m}"].mean())
            mlflow.log_metric(f"cv_{m}_std", cv[f"test_{m}"].std())
            for fold, value in enumerate(cv[f"test_{m}"]):        # per-fold values as a step series
                mlflow.log_metric(f"cv_{m}_fold", value, step=fold)

        # Artifact: the per-fold table as CSV
        fold_table = pd.DataFrame({m: cv[f"test_{m}"] for m in SCORING}).rename_axis("fold")
        mlflow.log_table(fold_table.reset_index(), "tables/cv_folds.json")

        # Plots: from out-of-fold predictions on the development set
        oof = cross_val_predict(pipeline, X_train, y_train, cv=folds, method="predict_proba", n_jobs=-1)[:, 1]
        pipeline.fit(X_train, y_train)
        fig1, fig2 = make_plots(pipeline, oof, run_name)
        mlflow.log_figure(fig1, "plots/roc_confusion_cv.png")
        mlflow.log_figure(fig2, "plots/feature_importance.png")
        plt.close(fig1); plt.close(fig2)

        # Model: the whole pipeline, with input/output schema and an example
        signature = infer_signature(X_train, pipeline.predict_proba(X_train)[:, 1])
        mlflow.sklearn.log_model(pipeline, name="model", signature=signature,
                                 input_example=X_train.head(3),
                                 skops_trusted_types=TRUSTED_TYPES)

        print(f"{run_name:<34} cv_roc_auc {cv['test_roc_auc'].mean():.3f} ± {cv['test_roc_auc'].std():.3f}")
        return run.info.run_id

## 4. Feature-set test (2 runs)
Same question as Step 2 — keep `ca`/`thal`/`slope` with missing indicators, or drop them? — but now both answers are **recorded**.

In [ ]:
feature_runs = {}
for drop in (False, True):
    name = f"feature-set: {'drop' if drop else 'keep'} sparse"
    feature_runs[drop] = log_full_run(
        make_pipeline(LogisticRegression(max_iter=1000), drop), name,
        tags={"stage": "feature_selection", "model_type": "logistic_regression"}, drop_sparse=drop)

def cv_auc(run_id):
    return mlflow.get_run(run_id).data.metrics["cv_roc_auc_mean"]

DROP_SPARSE = cv_auc(feature_runs[True]) > cv_auc(feature_runs[False])
print("Decision:", "DROP sparse columns" if DROP_SPARSE else "KEEP sparse columns with indicators")

## 5. Baseline runs (2 runs, default settings)

In [ ]:
baseline_runs = {
    "Logistic Regression": log_full_run(
        make_pipeline(LogisticRegression(max_iter=1000), DROP_SPARSE), "LR - baseline",
        tags={"stage": "baseline", "model_type": "logistic_regression"}, drop_sparse=DROP_SPARSE),
    "Random Forest": log_full_run(
        make_pipeline(RandomForestClassifier(random_state=SEED), DROP_SPARSE), "RF - baseline",
        tags={"stage": "baseline", "model_type": "random_forest"}, drop_sparse=DROP_SPARSE),
}

## 6. Tuning runs (parent + one child run per setting tried)
For each model, one **parent run** holds the whole search. Inside it:
- a **child run for every combination** in the grid, with its parameters and CV metrics — so nothing tried is lost;
- the search's full results table as a CSV artifact;
- one final child run, the **tuned model**, logged in full (plots + model).

In [ ]:
GRIDS = {
    "Logistic Regression": (
        LogisticRegression(max_iter=2000, solver="liblinear"),
        {"model__C": [0.01, 0.1, 1, 10], "model__l1_ratio": [0, 1]},
        "logistic_regression", "LR"),
    "Random Forest": (
        RandomForestClassifier(random_state=SEED),
        {"model__n_estimators": [200, 400], "model__max_depth": [4, 8, None],
         "model__min_samples_leaf": [1, 5, 10], "model__max_features": ["sqrt", 0.5]},
        "random_forest", "RF"),
}

tuned_runs, tuned_models = {}, {}
for name, (estimator, grid, model_type, short) in GRIDS.items():
    with mlflow.start_run(run_name=f"{short} - tuning"):
        common_context(DROP_SPARSE)
        mlflow.set_tags({"stage": "tuning", "model_type": model_type})
        mlflow.log_param("grid", str({k.replace("model__", ""): v for k, v in grid.items()}))

        search = GridSearchCV(make_pipeline(estimator, DROP_SPARSE), grid, cv=folds,
                              scoring=SCORING, refit="roc_auc", n_jobs=-1).fit(X_train, y_train)
        res = pd.DataFrame(search.cv_results_)

        # One child run per combination tried
        for i, params in enumerate(res["params"]):
            label = ", ".join(f"{k.replace('model__', '')}={v}" for k, v in params.items())
            with mlflow.start_run(run_name=f"{short} | {label}", nested=True):
                mlflow.set_tags({"stage": "tuning_candidate", "model_type": model_type})
                mlflow.log_params({k.replace("model__", ""): v for k, v in params.items()})
                for m in SCORING:
                    mlflow.log_metric(f"cv_{m}_mean", res.loc[i, f"mean_test_{m}"])
                    mlflow.log_metric(f"cv_{m}_std", res.loc[i, f"std_test_{m}"])

        # Search summary on the parent run
        summary = res[["params"] + [f"mean_test_{m}" for m in SCORING]].copy()
        summary["params"] = summary["params"].astype(str)
        mlflow.log_table(summary.sort_values("mean_test_roc_auc", ascending=False), "tables/grid_results.json")
        mlflow.log_metric("best_cv_roc_auc", search.best_score_)

        # The tuned model, logged in full as a child run
        tuned_models[name] = search.best_estimator_
        tuned_runs[name] = log_full_run(search.best_estimator_, f"{short} - tuned",
                                        tags={"stage": "tuned", "model_type": model_type},
                                        drop_sparse=DROP_SPARSE, nested=True)

## 7. Select the model (same rule as Step 2) and log test-set results
Rule: higher CV ROC-AUC wins, unless the gap is within one standard deviation — then the simpler Logistic Regression wins.
The test set is used once, for both tuned models; the results are added to their existing runs as `test_*` metrics.

In [ ]:
m = {name: mlflow.get_run(rid).data.metrics for name, rid in tuned_runs.items()}
gap = m["Random Forest"]["cv_roc_auc_mean"] - m["Logistic Regression"]["cv_roc_auc_mean"]
noise = max(m["Random Forest"]["cv_roc_auc_std"], m["Logistic Regression"]["cv_roc_auc_std"])
chosen = "Random Forest" if gap > noise else "Logistic Regression"
reason = (f"CV ROC-AUC gap RF-LR = {gap:+.3f}; noise (max std) = {noise:.3f}. "
          + ("RF clearly better." if chosen == "Random Forest" else "Within noise -> simpler model."))
print("Selected:", chosen, "|", reason)

test_rows = []
for name, model in tuned_models.items():
    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    scores = {"test_accuracy": accuracy_score(y_test, pred), "test_precision": precision_score(y_test, pred),
              "test_recall": recall_score(y_test, pred), "test_roc_auc": roc_auc_score(y_test, proba)}
    with mlflow.start_run(run_id=tuned_runs[name]):          # re-open the tuned run and add to it
        mlflow.log_metrics(scores)
        mlflow.set_tag("selected", str(name == chosen))
        if name == chosen:
            mlflow.set_tag("selection_reason", reason)
    test_rows.append({"model": name, **scores})
pd.DataFrame(test_rows).set_index("model").round(3)

## 8. Register the chosen model
The **Model Registry** is a catalogue of approved models. Registering creates version 1, 2, 3… of
`heart-disease-classifier`; the **alias `champion`** points at the version that should be served.
The API in Step 4 will simply load `models:/heart-disease-classifier@champion` — it never needs to know run IDs or file paths.

In [ ]:
client = MlflowClient()
version = mlflow.register_model(f"runs:/{tuned_runs[chosen]}/model", REGISTERED_MODEL)
client.set_registered_model_alias(REGISTERED_MODEL, "champion", version.version)
client.update_model_version(REGISTERED_MODEL, version.version,
                            description=f"{chosen}. {reason}")
print(f"Registered {REGISTERED_MODEL} version {version.version} as @champion ({chosen})")

In [ ]:
# Prove it works: load the champion by name and predict raw patient rows
champion = mlflow.sklearn.load_model(f"models:/{REGISTERED_MODEL}@champion")
preview = X_test.head(3).copy()
preview["disease_probability"] = champion.predict_proba(X_test.head(3))[:, 1].round(3)
preview

## 9. Query runs from code
Everything in the UI is also available as a DataFrame — useful for reports and, later, for CI checks.

In [ ]:
runs = mlflow.search_runs(experiment_names=[EXPERIMENT], order_by=["metrics.cv_roc_auc_mean DESC"])
runs = runs[runs["tags.stage"].isin(["baseline", "tuned"])]
cols = ["tags.mlflow.runName", "tags.stage", "metrics.cv_roc_auc_mean", "metrics.cv_roc_auc_std",
        "metrics.cv_recall_mean", "metrics.test_roc_auc", "metrics.test_recall"]
runs[cols].rename(columns=lambda c: c.split(".", 1)[1]).round(3)

## 10. Open the MLflow UI
In a **Terminal**, from the project root (with the virtual environment active):
```bash
mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5001
```
Then open **http://127.0.0.1:5001** in your browser. Stop it with **Ctrl + C**.

*(Port 5001, not MLflow's default 5000: on macOS, port 5000 is used by AirPlay Receiver.)*

**What to look at / screenshot for the report:**
1. The experiment's run list — expand the tuning parent runs to see all child runs.
2. Select the two `tuned` runs → **Compare** → parameters and metrics side by side.
3. Open a tuned run → **Artifacts** → `plots/` (ROC, confusion matrix, feature importance) and `model/`.
4. **Models** tab → `heart-disease-classifier` → the version with the `@champion` alias.